In [1]:
!uv pip install scikit-learn

Checked 1 package in 33ms


In [3]:
import torch
import torch.nn as nn

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [6]:
# Load dataset
wine = load_wine()

X = wine.data
y = wine.target


print(X.shape, y.shape)
print(wine.target_names)

print(X[:5])
print(y[:5])

(178, 13) (178,)
['class_0' 'class_1' 'class_2']
[[1.423e+01 1.710e+00 2.430e+00 1.560e+01 1.270e+02 2.800e+00 3.060e+00
  2.800e-01 2.290e+00 5.640e+00 1.040e+00 3.920e+00 1.065e+03]
 [1.320e+01 1.780e+00 2.140e+00 1.120e+01 1.000e+02 2.650e+00 2.760e+00
  2.600e-01 1.280e+00 4.380e+00 1.050e+00 3.400e+00 1.050e+03]
 [1.316e+01 2.360e+00 2.670e+00 1.860e+01 1.010e+02 2.800e+00 3.240e+00
  3.000e-01 2.810e+00 5.680e+00 1.030e+00 3.170e+00 1.185e+03]
 [1.437e+01 1.950e+00 2.500e+00 1.680e+01 1.130e+02 3.850e+00 3.490e+00
  2.400e-01 2.180e+00 7.800e+00 8.600e-01 3.450e+00 1.480e+03]
 [1.324e+01 2.590e+00 2.870e+00 2.100e+01 1.180e+02 2.800e+00 2.690e+00
  3.900e-01 1.820e+00 4.320e+00 1.040e+00 2.930e+00 7.350e+02]]
[0 0 0 0 0]


### Split the dataset into 
- Training 80%
- Testing 20%

In [13]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

print(X_train[:5])
print(y_train[:5])

(142, 13) (142,)
(36, 13) (36,)
[[1.434e+01 1.680e+00 2.700e+00 2.500e+01 9.800e+01 2.800e+00 1.310e+00
  5.300e-01 2.700e+00 1.300e+01 5.700e-01 1.960e+00 6.600e+02]
 [1.253e+01 5.510e+00 2.640e+00 2.500e+01 9.600e+01 1.790e+00 6.000e-01
  6.300e-01 1.100e+00 5.000e+00 8.200e-01 1.690e+00 5.150e+02]
 [1.237e+01 1.070e+00 2.100e+00 1.850e+01 8.800e+01 3.520e+00 3.750e+00
  2.400e-01 1.950e+00 4.500e+00 1.040e+00 2.770e+00 6.600e+02]
 [1.348e+01 1.670e+00 2.640e+00 2.250e+01 8.900e+01 2.600e+00 1.100e+00
  5.200e-01 2.290e+00 1.175e+01 5.700e-01 1.780e+00 6.200e+02]
 [1.307e+01 1.500e+00 2.100e+00 1.550e+01 9.800e+01 2.400e+00 2.640e+00
  2.800e-01 1.370e+00 3.700e+00 1.180e+00 2.690e+00 1.020e+03]]
[2 2 1 2 0]


In [11]:
142/178, 36/178

(0.797752808988764, 0.20224719101123595)

In [14]:

# Normalize features
scaler = StandardScaler() # (x-mean)/std

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [15]:
# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [16]:
# Number of features
input_size = X_train.shape[1] # 13

# Number of classes
num_classes = len(torch.unique(y_train)) # 3

In [17]:
# Logistic Regression Model
class LogisticRegression(nn.Module):

    def __init__(self, input_size, num_classes):
        super().__init__()

        self.linear = nn.Linear(input_size,num_classes)

    def forward(self, x):

        logits = self.linear(x)

        return logits

In [18]:
# Create model
model = LogisticRegression(input_size, num_classes) # 13, 3

###  Cross entropy ( use this for multi-class classification) and use BCELoss (for regression task)
- Cross-entropy loss, or log loss, measures the performance of a classification model 
- whose output is a probability value between 0 and 1. 
-  -> loss increases as the predicted probability diverges from the actual label

In [19]:
# Loss function
loss_fn = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.SGD(model.parameters(),lr=0.01)

In [20]:
# Training
epochs = 1000

for epoch in range(epochs):

    # Forward pass
    outputs = model(X_train)

    # Compute loss
    loss = loss_fn(outputs, y_train)

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    # Zero gradients
    optimizer.zero_grad()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.8182
Epoch 100, Loss: 0.3576
Epoch 200, Loss: 0.2483
Epoch 300, Loss: 0.1974
Epoch 400, Loss: 0.1671
Epoch 500, Loss: 0.1467
Epoch 600, Loss: 0.1319
Epoch 700, Loss: 0.1205
Epoch 800, Loss: 0.1114
Epoch 900, Loss: 0.1040


In [21]:
# Evaluation
with torch.no_grad():

    outputs = model(X_test)

    # Predicted class
    _, predictions = torch.max(outputs, 1)

    accuracy = (predictions == y_test).float().mean()

    print("\nAccuracy:", accuracy.item())


Accuracy: 1.0


### Make a prediction

In [27]:
X_test[21]

tensor([ 0.1602, -1.1962, -2.3752, -1.2994, -1.5373,  1.0873,  1.1771, -0.8452,
         1.1554,  0.1044,  0.7014,  0.8160, -0.7731])

In [28]:
# make a prediction on a single sample
with torch.no_grad():

    sample = X_test[21].unsqueeze(0) # Add batch dimension

    output = model(sample)

    _, predicted_class = torch.max(output, 1)

    print("\nPredicted class:", predicted_class.item())


Predicted class: 1


In [30]:
!uv pip install pandas

Resolved 4 packages in 599ms                                         
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/9.88 MiB            
⠙ Preparing packages... (0/1)------------------- 16.00 KiB/9.88 MiB          
⠙ Preparing packages... (0/1)------------------- 32.00 KiB/9.88 MiB          
⠙ Preparing packages... (0/1)------------------- 48.00 KiB/9.88 MiB          
⠙ Preparing packages... (0/1)------------------- 62.66 KiB/9.88 MiB          
⠙ Preparing packages... (0/1)------------------- 78.66 KiB/9.88 MiB          
⠙ Preparing packages... (0/1)------------------- 94.66 KiB/9.88 MiB          
⠙ Preparing packages... (0/1)------------------- 110.66 KiB/9.88 MiB         
⠙ Preparing packages... (0/1)------------------- 126.66 KiB/9.88 MiB         
⠹ Preparing packages... (0/1)------------------- 142.66 KiB/9.88 MiB         
⠹ Preparing packages... (0/1)------------------- 142.66 KiB/9.88 MiB 

In [31]:
import pandas as pd

pd.DataFrame(X_test.numpy(), columns=wine.feature_names)

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,0.808733,0.637319,0.715786,-1.241280,1.065567,0.646637,1.027242,-1.549321,0.089361,0.018252,0.015517,1.066134,0.365487
1,1.506217,1.461953,0.284492,-0.166513,0.723081,0.882684,0.647481,-0.532235,-0.615595,0.078527,-0.370294,1.024444,1.145552
2,-0.060063,0.382910,1.218962,0.443490,-0.304379,-1.178796,-1.501170,1.267226,-1.475296,-0.197015,-0.798972,-0.393023,-0.447771
3,0.918862,-0.766315,1.218962,0.879206,0.038108,1.118731,1.247104,-0.610472,1.327331,0.276573,1.001477,0.162846,1.826033
4,-0.745310,-1.055814,-1.584448,0.036821,-1.537330,-0.281816,-0.002111,-0.766947,-0.976669,-0.162572,0.701402,1.232895,-0.746520
5,1.616346,-0.397861,1.290844,0.153012,1.339556,0.804002,1.137173,-0.297523,0.622375,0.491840,0.487063,0.079466,1.809436
6,-1.198063,0.926818,-1.296919,-0.137465,-0.920855,-0.454917,-0.361885,0.015427,0.450435,-1.626389,-0.113087,0.635335,-0.567271
7,0.515056,1.347908,0.428257,1.024445,0.106605,-0.769647,-1.251327,0.484852,-0.340490,0.965428,-1.099047,-1.435279,0.050142
8,-1.675289,-0.897905,1.218962,0.153012,-0.441373,0.709583,0.917311,-0.610472,1.516465,-1.036557,0.015517,0.927167,-0.182218
9,0.466110,0.163592,-0.038978,0.153012,-0.783860,-1.399107,-1.501170,0.015427,-1.664431,0.233519,-1.099047,-0.170675,0.149725
